In [0]:
#dbutils.library.restartPython()


In [0]:
# ============================================================
# 0. Libraries
# ============================================================

import pyspark.sql.functions as f
from common_utils.bronze import read_raw, bronze_ingestor
from common_utils.logging import get_logger
from datetime import datetime
import json

# ============================================================
# 1. Setting up logger
# ============================================================

logger= get_logger("ds2b_s3")

logger.info("Session started for s3_sales")


# ============================================================
# 2. Config path
# ============================================================

dbutils.widgets.text("path", "")
config_path = dbutils.widgets.get("path")
logger.info("setting path for config files")

# ============================================================
# 3. Reading config path
# ============================================================
logger.info("Reading configuration from %s", config_path)
with open(config_path, "r") as file:
    config = json.load(file)

logger.info("configuration loaded succesfully")


# =========================================
# 2. Read Config
# =========================================
with open(config_path, "r") as file:
  config = json.load(file) 


logger.info('Session created')
# As I have setup the required stuff now Im going to make and address the sources


logger.info("soruces created")
# ============================================================
# 4. RUN DATE
# ============================================================

run_date = datetime.now().strftime("%Y-%m-%d")
logger.info("[%s] date data will be processed", run_date)

logger.info("Bronze ingestion run date: %s", run_date)

# ============================================================
# 5. BASE RAW PATH
# ============================================================

base_raw_path = "/Volumes/retail_data_platform/bronze/raw_data/"

logger.info("Base raw path: %s", base_raw_path)


# ============================================================
# 6. Processing the bronze
# ============================================================

for source_name, source_config in config.items():
    raw_folder = source_config["raw_folder"]
    file_format = source_config["file_format"]
    read_options = source_config.get("options", {})
    target_table = source_config["target_table"]
    mode = source_config["mode"]
    
    # --------------------------------------------------------
    # Build raw path
    # --------------------------------------------------------
    raw_path = (
        f"{base_raw_path}"
        f"{raw_folder}/"f"load_date={run_date}"
    )

    logger.info(
        "%s reading from path %s",
        source_name,
        raw_path
    )

    df = read_raw(
        spark,
        raw_path,
        file_format=file_format,
        options=read_options
    )

    rows = bronze_ingestor(
        df,
        mode=mode,
        target_table=target_table
    )

    logger.info(
        "%s rows ingested into the bronze %s",
        rows,
        source_name
    )

logger.info("Bronze layer completed")

